# Kiểm tra các dataset đặc trưng (token / vector) đã trích (chỉ CPU, chỉ đọc)

Notebook in báo cáo cho **mọi dataset được Add Input**:
- số file, dung lượng, loại file, thư mục con, giao thức (`action_start` / `official`, đoán từ đường dẫn);
- với vài file mẫu: tên các mảng, **shape**, dtype, số NaN / Inf, trung bình và độ lệch chuẩn;
- khoá id của clip (`clip_id`, `narration_id`, ...), số dòng và số id khác nhau;
- giữa các dataset: tổng số clip mỗi giao thức, **id nào bị trùng** giữa hai dataset cùng giao thức, và độ phủ so với các segment
  train / val của EK100 (nếu id là narration id).

**Cài đặt:** Accelerator **None**, Internet On. *Add Input* cả 10 dataset `ek100-tok-*`. Dataset của tài khoản khác phải để
**Public**, nếu không sẽ gặp lỗi mount. Kết quả: in ra màn hình và lưu ở `feature_datasets_report.json`.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
R = "/kaggle/working/repo"
ANN = "/kaggle/working/ek100-ann"
!find /kaggle/input -maxdepth 4 -type d | head -60


In [ ]:
import glob
idx = glob.glob("/kaggle/input/**/val_clips.csv", recursive=True) + glob.glob("/kaggle/input/**/train_clips.csv", recursive=True)
IDX = " ".join(idx[:4])
!python $R/scripts/inspect_feature_datasets.py --root /kaggle/input --train_csv $ANN/EPIC_100_train.csv \
    --val_csv $ANN/EPIC_100_validation.csv --clip_index $IDX --out /kaggle/working/feature_datasets_report.json


In [ ]:
# Clip index / class maps / exp_code / shard folders of each dataset (to map the per-dataset clip_id to segments)
import glob, os, json, pandas as pd
import numpy as np
for d in sorted(glob.glob("/kaggle/input/datasets/*/*")):
    print("=" * 100, "\n", d)
    print("exp_code:", sorted(os.listdir(os.path.join(d, "exp_code"))) if os.path.isdir(os.path.join(d, "exp_code")) else "-")
    print("shard dirs:", sorted(os.path.relpath(s, d) for s in glob.glob(d + "/**/shard*", recursive=True) if os.path.isdir(s)))
    for f in sorted(glob.glob(d + "/**/*.csv", recursive=True) + glob.glob(d + "/**/*.json", recursive=True))[:10]:
        print("--", os.path.relpath(f, d), os.path.getsize(f), "bytes")
        if f.endswith(".csv"):
            df = pd.read_csv(f)
            print("  ", df.shape, list(df.columns)[:20])
            print(df.head(3).to_string()[:900])
        else:
            txt = open(f).read()
            try:
                j = json.loads(txt)
                if isinstance(j, dict):
                    print("   keys:", list(j)[:15], "| sizes:", {k: len(v) for k, v in j.items() if hasattr(v, "__len__") and not isinstance(v, str)})
                else:
                    print("  ", txt[:300])
            except Exception:
                print("  ", txt[:300])
    # duplicates inside the dataset
    ids = {}
    for f in glob.glob(d + "/**/*.npz", recursive=True):
        proto = "official" if "official" in f.split(d)[1] and "action_start" not in f.split(d)[1] else "action_start" if "action_start" in f else "?"
        with np.load(f) as z:
            ids.setdefault(proto, []).extend(z["clip_id"].tolist())
    print("rows / unique clip_id per protocol:", {k: (len(v), len(set(v))) for k, v in ids.items()})


Gửi lại **toàn bộ output** của cell cuối (và cell liệt kê thư mục), hoặc tải file `feature_datasets_report.json` trong tab Output.